# 1. import

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
from tqdm import tqdm
import os

# 2. data load

In [ ]:
data_path = "C:/Users/seung/OneDrive/바탕 화면/2025_ML/data"

train_file_path = os.path.join(data_path, 'train.csv')
test_file_path = os.path.join(data_path, 'test.csv')
sample_submission_file_path = os.path.join(data_path, 'sample_submission.csv')

train = pd.read_csv(train_file_path, encoding='utf-8-sig')
test = pd.read_csv(test_file_path, encoding='utf-8-sig')

print("Train shape:", train.shape)
print("Test shape:", test.shape)

# 3. main

In [ ]:
# ==========================================
# 1. 자모 분해/결합 유틸리티 (필수)
# ==========================================
# (한글 처리를 위해 기존 코드의 유틸리티를 그대로 사용합니다)
BASE_CODE = 0xAC00
CHOSUNG  = 21 * 28
JUNGSUNG = 28
CHOSUNG_LIST = ['ㄱ','ㄲ','ㄴ','ㄷ','ㄸ','ㄹ','ㅁ','ㅂ','ㅃ','ㅅ','ㅆ','ㅇ','ㅈ','ㅉ','ㅊ','ㅋ','ㅌ','ㅍ','ㅎ']
JUNGSUNG_LIST = ['ㅏ','ㅐ','ㅑ','ㅒ','ㅓ','ㅔ','ㅕ','ㅖ','ㅗ','ㅘ','ㅙ','ㅚ','ㅛ','ㅜ','ㅝ','ㅞ','ㅟ','ㅠ','ㅡ','ㅢ','ㅣ']
JONGSUNG_LIST = ['', 'ㄱ','ㄲ','ㄳ','ㄴ','ㄵ','ㄶ','ㄷ','ㄹ','ㄺ','ㄻ','ㄼ','ㄽ','ㄾ','ㄿ','ㅀ','ㅁ','ㅂ','ㅄ','ㅅ','ㅆ','ㅇ','ㅈ','ㅊ','ㅋ','ㅌ','ㅍ','ㅎ']

def to_jamo_list(text):
    """문자열을 자모 리스트로 분해 (예: '부산' -> ['ㅂ','ㅜ','ㅅ','ㅏ','ㄴ',''])"""
    result = []
    for ch in text:
        code = ord(ch) - BASE_CODE
        if code < 0 or code > (0xD7A3 - 0xAC00):
            result.append(ch) # 한글 아님
            continue
        cho  = code // CHOSUNG
        jung = (code % CHOSUNG) // JUNGSUNG
        jong = (code % CHOSUNG) % JUNGSUNG
        result.extend([CHOSUNG_LIST[cho], JUNGSUNG_LIST[jung], JONGSUNG_LIST[jong]])
    return result

def from_jamo_list(jamo_list):
    """자모 리스트를 문자열로 결합"""
    result = []
    # 3개씩(초,중,종) 묶어서 처리. 단, 완벽한 복원을 위해 Queue 방식 사용 추천되나
    # 여기서는 간단히 3개 단위 처리를 가정 (길이가 3의 배수여야 함)
    # 실제 ML 예측시 종성이 없는 경우 ''(빈문자)를 예측하도록 학습됨
    i = 0
    while i < len(jamo_list) - 2:
        cho, jung, jong = jamo_list[i], jamo_list[i+1], jamo_list[i+2]
        if cho in CHOSUNG_LIST and jung in JUNGSUNG_LIST and jong in JONGSUNG_LIST:
            cho_i = CHOSUNG_LIST.index(cho)
            jung_i = JUNGSUNG_LIST.index(jung)
            jong_i = JONGSUNG_LIST.index(jong)
            code = BASE_CODE + (cho_i * CHOSUNG) + (jung_i * JUNGSUNG) + jong_i
            result.append(chr(code))
            i += 3
        else:
            result.append(cho) # 복원 실패시 그냥 글자 추가
            i += 1
    return "".join(result)

# ==========================================
# 2. 데이터셋 생성 (Sliding Window)
# ==========================================
def create_dataset(df, window_size=3):
    """
    입력 텍스트를 윈도우 단위로 잘라서 X, y 데이터셋 생성
    X: [앞_자모, 현재_자모, 뒤_자모] (문맥)
    y: [정답_자모]
    """
    data_X = []
    data_y = []
    
    # 패딩용 토큰
    PAD = '<PAD>'
    
    print("Generating Dataset...")
    for idx, row in tqdm(df.iterrows(), total=len(df)):
        # 입력과 정답의 자모 길이가 같아야 학습 가능 (간단한 가정을 위해 길이 맞춤)
        # 실제로는 alignment가 필요하지만, 여기서는 길이가 같은 데이터만 학습에 사용하거나
        # 길이가 다르면 과감히 패스하는 전략을 사용 (Scikit-learn의 한계)
        
        in_jamos = to_jamo_list(row['input'])
        out_jamos = to_jamo_list(row['output'])
        
        # 길이가 안 맞으면 학습에서 제외 (노이즈가 심해 구조가 깨진 경우)
        # *Tip: 성능을 높이려면 여기서 레벤슈타인 정렬(Alignment)을 해서 짝을 맞춰줘야 함
        if len(in_jamos) != len(out_jamos):
            continue
            
        # 앞뒤 패딩 추가
        padded_in = [PAD] * (window_size // 2) + in_jamos + [PAD] * (window_size // 2)
        
        for i in range(len(in_jamos)):
            # 윈도우 추출 (예: window=5면 앞2개, 본인, 뒤2개)
            window = padded_in[i : i + window_size]
            target = out_jamos[i]
            
            data_X.append(window)
            data_y.append(target)
            
    return data_X, data_y

# ==========================================
# 3. 메인 파이프라인
# ==========================================

# 1) 데이터셋 생성 (자모 단위 분류 문제로 변환)
# Window Size: 5 (앞 2개, 본인, 뒤 2개 -> 총 5개의 문맥을 보고 판단)
WINDOW_SIZE = 5 
X_raw, y_raw = create_dataset(train, window_size=WINDOW_SIZE)

print(f"Total Samples (Characters): {len(X_raw)}")

# 2) 문자를 숫자로 변환 (Label Encoding)
# 모든 가능한 자모 + 특수문자 수집
all_tokens = set(np.array(X_raw).flatten()) | set(y_raw)
token_to_id = {t: i for i, t in enumerate(sorted(list(all_tokens)))}
id_to_token = {i: t for t, i in token_to_id.items()}

# 인코딩 함수
def encode_features(X_list, token_map):
    encoded = []
    unknown_token = token_map.get('<PAD>', 0)
    for sample in X_list:
        encoded.append([token_map.get(t, unknown_token) for t in sample])
    return np.array(encoded)

print("Encoding Features...")
# [최적화 1] 데이터 타입을 int16으로 줄여서 메모리 절약 (매우 중요!)
X_encoded = encode_features(X_raw, token_to_id).astype(np.int16)
y_encoded = np.array([token_to_id[t] for t in y_raw]).astype(np.int16)

# 3) 모델 학습 (Random Forest)
print("Training RandomForest Classifier...")
# [최적화 2] 메모리 절약을 위한 파라미터 튜닝
model = RandomForestClassifier(
    n_estimators=50,       # 나무 개수
    n_jobs=1,              # [중요] 병렬 처리 끄기 (-1 -> 1). 속도는 느려지지만 메모리는 안 터짐
    max_depth=20,          # [중요] 트리 깊이 제한 (무한대 -> 20). 메모리 폭증 방지
    min_samples_split=10,  # 과적합 방지 및 메모리 절약
    random_state=42, 
    verbose=1
)
model.fit(X_encoded, y_encoded)
print("Training Done!")

# 4) 성능 검증 (선택)
train_acc = accuracy_score(y_encoded, model.predict(X_encoded))
print(f"Train Accuracy: {train_acc:.4f}")

# ==========================================
# 4. 추론 (Test 복원)
# ==========================================
def restore_text(text, model, token_map, inv_token_map, window_size=5):
    in_jamos = to_jamo_list(text)
    PAD = '<PAD>'
    padded_in = [PAD] * (window_size // 2) + in_jamos + [PAD] * (window_size // 2)
    
    windows = []
    for i in range(len(in_jamos)):
        win = padded_in[i : i + window_size]
        windows.append(win)
    
    # 인코딩 및 예측
    X_test = encode_features(windows, token_map)
    y_pred = model.predict(X_test)
    
    # 자모 복원
    restored_jamos = [inv_token_map[pid] for pid in y_pred]
    
    # 문자열 결합
    return from_jamo_list(restored_jamos)

print("Restoring Test Data...")
outputs = []
for idx, row in tqdm(test.iterrows(), total=len(test)):
    restored = restore_text(row['input'], model, token_to_id, id_to_token, window_size=WINDOW_SIZE)
    outputs.append(restored)



# 4. save

In [ ]:
submission = pd.read_csv(sample_submission_file_path, encoding='utf-8-sig')
submission['output'] = outputs
submission.to_csv('submission_sklearn.csv', index=False)
print("Done! Saved to submission_sklearn.csv")